In [ ]:
import os
import json
import pandas as pd
import re
import numpy as np


# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from googleapiclient.discovery import build
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe

In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [ ]:
# Open files

forms_data = gc.open_by_key('1947Wx86ZtNWQSaqcYVSXv_3WLvIA0p6u_Ol1DZ8GmX8')
forms_data = forms_data.get_worksheet(0)
forms_data = get_as_dataframe(forms_data)

influencers_posts = gc.open_by_key('1o96u5EXkqhtxGdEqaUGYX4Us2HGnHfkVBLJIobqcma8')
influencers_posts = influencers_posts.get_worksheet(0)
influencers_posts = get_as_dataframe(influencers_posts)

influencers_comments = gc.open_by_key('1shH8-PpUBTEuS7Izy4uTgmEcOHF-tdk_DbJR1ifXqJA')
influencers_comments = influencers_comments.get_worksheet(0)
influencers_comments = get_as_dataframe(influencers_comments)

posts_data = gc.open_by_key('1CtvNfYM5Jp_kuriycsYMAMzCQYW0pFxqvGmOD0O4n80')
posts_data = posts_data.worksheet('tt_data_post_post_max')
posts_data = get_as_dataframe(posts_data)

posts_comments = gc.open_by_key('1BD4OoVfXZHI6p5kJ6KmLAMsPfpQ86MjdNdVoPPWhgkg')
posts_comments = posts_comments.get_worksheet(0)
posts_comments = get_as_dataframe(posts_comments)

In [ ]:
# Clean databases
forms_data = forms_data.fillna(0)
influencers_posts = influencers_posts.fillna(0)
influencers_comments = influencers_comments.fillna(0)
posts_data = posts_data.fillna(0)
posts_comments = posts_comments.fillna(0)

/tmp/ipykernel_788/135062332.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posts_comments = posts_comments.fillna(0)


In [ ]:
# Create influencers baseline for negative sentiment
influencers_comments = influencers_comments.rename(columns={'username': 'commenter'})
influencers_comments = influencers_comments.merge(influencers_posts[["video_url", "username"]], on='video_url', how='left')
neg_sent_baselines = influencers_comments.groupby('username').agg(
    detrator_count=('classification', lambda x: x.isin(['detrator', 'Detractor', 'detractor']).sum()),
    total_rows=('classification', 'count')
).reset_index()


neg_sent_baselines['negative_influencer_baseline'] = neg_sent_baselines['detrator_count'] / neg_sent_baselines['total_rows']

In [ ]:
# Create influencers baseline for engagement rate
# 1. Create the "total_interactions" column
influencers_posts['total_interactions'] = influencers_posts['likes'] + influencers_posts['comments'] + influencers_posts['shares']

# 2. Remove rows where play_count is 0
influencers_posts = influencers_posts[influencers_posts['play_count'] != 0]

# 3. Aggregate (sum) total_interactions and play_count by username
er_baselines = influencers_posts.groupby('username').agg(
    total_interactions=('total_interactions', 'sum'),
    play_count=('play_count', 'sum')
).reset_index()

# 4. Create the "er_influencer_baseline" column
er_baselines['er_influencer_baseline'] = (
    er_baselines['total_interactions'] / er_baselines['play_count']
)

In [ ]:
# Consolidate sentiment for each published post
posts_comments = posts_comments.rename(columns={'username': 'commenter'})
posts_comments = posts_comments.merge(posts_data[["video_url", "username"]], on='video_url', how='left')
sentiment_posts = posts_comments.groupby(['video_url', 'username']).agg(
    positive_count=('classification', lambda x: x.isin(['promotor', 'Promoter', 'promoter', 'Promotor']).sum()),
    negative_count=('classification', lambda x: x.isin(['detrator', 'Detractor', 'detractor', 'Detractor']).sum()),
    neutral_count=('classification', lambda x: x.isin(['neutro', 'neutral', 'Neutro', 'Neutral']).sum()),
).reset_index()
# 2. Sum the three sentiment columns to get the total
sentiment_posts['total_sentiments'] = (
    sentiment_posts['positive_count'] +
    sentiment_posts['negative_count'] +
    sentiment_posts['neutral_count']
)

# 3. Calculate the percentage of each sentiment over the total
sentiment_posts['positive_percentage'] = sentiment_posts['positive_count'] / sentiment_posts['total_sentiments']
sentiment_posts['negative_percentage'] = sentiment_posts['negative_count'] / sentiment_posts['total_sentiments']
sentiment_posts['neutral_percentage'] = sentiment_posts['neutral_count'] / sentiment_posts['total_sentiments']

In [ ]:
sentiment_posts

,video_url,username,positive_count,negative_count,neutral_count,total_sentiments,positive_percentage,negative_percentage,neutral_percentage
0,https://www.tiktok.com/@___lauraortega/video/7...,___lauraortega,65,5,6,76,0.855263,0.065789,0.078947
1,https://www.tiktok.com/@___lauraortega/video/7...,___lauraortega,47,6,8,61,0.770492,0.098361,0.131148
2,https://www.tiktok.com/@_beluuum/video/7632146...,_beluuum,2,1,0,3,0.666667,0.333333,0.000000
3,https://www.tiktok.com/@_beluuum/video/7657010...,_beluuum,5,2,2,9,0.555556,0.222222,0.222222
4,https://www.tiktok.com/@_kiara.c__/video/76116...,_kiara.c__,8,0,0,8,1.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
288,https://www.tiktok.com/@vickylsmakeuphair/vide...,vickylsmakeuphair,2,2,0,4,0.500000,0.500000,0.000000
289,https://www.tiktok.com/@virgidemo/video/764646...,virgidemo,19,1,0,20,0.950000,0.050000,0.000000
290,https://www.tiktok.com/@yanellacamacho/video/7...,yanellacamacho,5,0,0,5,1.000000,0.000000,0.000000
291,https://www.tiktok.com/@yanellacamacho/video/7...,yanellacamacho,2,1,0,3,0.666667,0.333333,0.000000


In [ ]:
# Prepare datasets for join

posts_data = posts_data.rename(columns={'username': 'influencer'})
posts_data = posts_data.rename(columns={'video_url': 'url'})
forms_data = forms_data.rename(columns={'Link of Post': 'url'})
sentiment_posts = sentiment_posts.rename(columns={'video_url': 'url'})
neg_sent_baselines = neg_sent_baselines.rename(columns={'username': 'influencer'})
er_baselines = er_baselines.rename(columns={'username': 'influencer'})

In [ ]:
# Join datasets
tiktok_influencers_final = posts_data
tiktok_influencers_final = tiktok_influencers_final.merge(forms_data, on='url', how='left')
tiktok_influencers_final = tiktok_influencers_final.merge(sentiment_posts, on='url', how='left')
tiktok_influencers_final = tiktok_influencers_final.merge(neg_sent_baselines, on='influencer', how='left')
tiktok_influencers_final = tiktok_influencers_final.merge(er_baselines, on='influencer', how='left')
tiktok_influencers_final

,url,influencer,run_datetime,aweme_id,likes,comment_count,share_count,views,saves,download_count,...,total_sentiments,positive_percentage,negative_percentage,neutral_percentage,detrator_count,total_rows,negative_influencer_baseline,total_interactions,play_count,er_influencer_baseline
0,https://www.tiktok.com/@michelle.iman/video/74...,michelle.iman,2026-05-20 18:44:28,7489491180615470342,3275,32,32,50162,24,2,...,30.0,0.933333,0.066667,0.000000,323.0,1533.0,0.210698,255763.0,2699291.0,0.094752
1,https://www.tiktok.com/@carotrippar/video/7489...,carotrippar,2026-05-20 18:44:04,7489561985940491542,27877,173,228,168394,298,17,...,131.0,0.954198,0.045802,0.000000,2141.0,4776.0,0.448283,637620.0,7364556.0,0.086580
2,https://www.tiktok.com/@danielacelis12/video/7...,danielacelis12,2026-05-20 18:40:33,7491078442088140087,114477,477,463,1673062,1431,88,...,204.0,0.725490,0.274510,0.000000,2281.0,7989.0,0.285518,2824293.0,34873713.0,0.080986
3,https://www.tiktok.com/@agustinbattioni/video/...,agustinbattioni,2026-05-20 18:40:20,7491824069449420038,399,3,6,22720,14,1,...,1.0,1.000000,0.000000,0.000000,50.0,1008.0,0.049603,248497.0,2559981.0,0.097070
4,https://www.tiktok.com/@agustinbattioni/video/...,agustinbattioni,2026-05-20 18:40:20,7491824069449420038,399,3,6,22720,14,1,...,1.0,1.000000,0.000000,0.000000,50.0,1008.0,0.049603,248497.0,2559981.0,0.097070
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,https://www.tiktok.com/@estefaniaacuna/video/7...,estefaniaacuna,2026-08-07 8:41:01,7647345627524844818,871,7,28,61381,79,1,...,3.0,1.000000,0.000000,0.000000,58.0,231.0,0.251082,746973.0,8791879.0,0.084962
311,https://www.tiktok.com/@kariinacamposs/video/7...,kariinacamposs,2026-08-07 8:41:09,7637943869375515924,168,2,1,8304,1,1,...,1.0,1.000000,0.000000,0.000000,9.0,187.0,0.048128,42225.0,547490.0,0.077125
312,https://www.tiktok.com/@kariinacamposs/video/7...,kariinacamposs,2026-08-07 8:41:17,7668514054863981844,1183,16,39,54681,94,5,...,8.0,0.625000,0.375000,0.000000,9.0,187.0,0.048128,42225.0,547490.0,0.077125
313,https://www.tiktok.com/@sisoymachis/video/7670...,sisoymachis,2026-08-07 8:41:24,7670347780652682527,8033,60,80,56983,226,0,...,44.0,0.522727,0.159091,0.318182,19.0,271.0,0.070111,361242.0,2660455.0,0.135782


In [ ]:
# Create Organic_id
tiktok_influencers_final["Organic_ID"] = tiktok_influencers_final["aweme_id"]
tiktok_influencers_final['Organic_ID'] = tiktok_influencers_final['Organic_ID'].astype('string')

In [ ]:
# Rename columns
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'desc': 'copy'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Published date': 'date_published'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Plataform': 'platform'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'comment_count': 'comments'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'share_count': 'shares'})
tiktok_influencers_final = tiktok_influencers_final.rename(columns={'Marca': 'brand'})


In [ ]:
# Crear columnas

tiktok_influencers_final["content_type"] = "Influencers"
tiktok_influencers_final["format"] = "Video"
tiktok_influencers_final["total_interactions"] = tiktok_influencers_final["likes"] + tiktok_influencers_final["comments"] + tiktok_influencers_final["shares"] + tiktok_influencers_final["saves"]
tiktok_influencers_final["engagement_rate"] = (
    tiktok_influencers_final["total_interactions"]
    .div(tiktok_influencers_final["views"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
tiktok_influencers_final["positive_comments"] = tiktok_influencers_final["positive_percentage"] * tiktok_influencers_final["comments"]
tiktok_influencers_final["negative_comments"] = tiktok_influencers_final["negative_percentage"] * tiktok_influencers_final["comments"]
tiktok_influencers_final["neutral_comments"] = tiktok_influencers_final["neutral_percentage"] * tiktok_influencers_final["comments"]
tiktok_influencers_final["delta_eng_rate"] = tiktok_influencers_final["engagement_rate"] - tiktok_influencers_final["er_influencer_baseline"]
tiktok_influencers_final["delta_neg_sentiment"] = tiktok_influencers_final["negative_percentage"] - tiktok_influencers_final["negative_influencer_baseline"]



In [ ]:
# Apply boosting criteria

def classify_boost(row):
    if row['delta_eng_rate'] > 0 and row['delta_neg_sentiment'] <= 0:
        return 'Boost'
    elif row['delta_eng_rate'] > 0 and row['delta_neg_sentiment'] > 0:
        return 'Review'
    else:
        return 'No Boost'

tiktok_influencers_final['Accionable'] = tiktok_influencers_final.apply(classify_boost, axis=1)

In [ ]:
# Seleccionar y ordenar columnas

cols = [
    "url",
    "copy",
    "date_published",
    "platform",
    "format",
    "influencer",
    "Country",
    "Organic_ID",
    "brand",
    "content_type",
    "views",
    "likes",
    "comments",
    "shares",
    "saves",
    "total_interactions",
    "engagement_rate",
    "positive_percentage",
    "negative_percentage",
    "neutral_percentage",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "er_influencer_baseline",
    "negative_influencer_baseline",
    "Accionable",
    "run_datetime"
]

tiktok_influencers_final = tiktok_influencers_final[cols]
tiktok_influencers_final = tiktok_influencers_final.reindex(columns=cols)

In [ ]:
# Adjust date
tiktok_influencers_final["date_published"] = pd.to_datetime(
    tiktok_influencers_final["date_published"],
    format="mixed",
    errors="coerce",
).dt.date

In [ ]:
tiktok_influencers_final

,url,copy,date_published,platform,format,influencer,Country,Organic_ID,brand,content_type,...,positive_percentage,negative_percentage,neutral_percentage,positive_comments,negative_comments,neutral_comments,er_influencer_baseline,negative_influencer_baseline,Accionable,run_datetime
0,https://www.tiktok.com/@michelle.iman/video/74...,Fuimos al nuevo lanzamiento de @HuggiesArgenti...,2025-04-04,TikTok,Video,michelle.iman,Argentina,7489491180615470342,Huggies,Influencers,...,0.933333,0.066667,0.000000,29.866667,2.133333,0.000000,0.094752,0.210698,No Boost,2026-05-20 18:44:28
1,https://www.tiktok.com/@carotrippar/video/7489...,Que emoción poder contarles que @huggiesarg va...,2025-04-04,TikTok,Video,carotrippar,Argentina,7489561985940491542,Huggies,Influencers,...,0.954198,0.045802,0.000000,165.076336,7.923664,0.000000,0.086580,0.448283,Boost,2026-05-20 18:44:04
2,https://www.tiktok.com/@danielacelis12/video/7...,@huggiesargentina ❤️#TeBancamosBebe,2025-04-08,TikTok,Video,danielacelis12,Argentina,7491078442088140087,Huggies,Influencers,...,0.725490,0.274510,0.000000,346.058824,130.941176,0.000000,0.080986,0.285518,No Boost,2026-05-20 18:40:33
3,https://www.tiktok.com/@agustinbattioni/video/...,Acompáñenme al espectacular evento de @Huggies...,2025-04-10,TikTok,Video,agustinbattioni,Argentina,7491824069449420038,Huggies,Influencers,...,1.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.097070,0.049603,No Boost,2026-05-20 18:40:20
4,https://www.tiktok.com/@agustinbattioni/video/...,Acompáñenme al espectacular evento de @Huggies...,2025-04-10,TikTok,Video,agustinbattioni,Argentina,7491824069449420038,Huggies,Influencers,...,1.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.097070,0.049603,No Boost,2026-05-20 18:40:20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,https://www.tiktok.com/@estefaniaacuna/video/7...,Cosas que volvería a comprar como mamá primeri...,2026-06-04,TikTok,Video,estefaniaacuna,Costa Rica,7647345627524844818,Huggies,Influencers,...,1.000000,0.000000,0.000000,7.000000,0.000000,0.000000,0.084962,0.251082,No Boost,2026-08-07 8:41:01
311,https://www.tiktok.com/@kariinacamposs/video/7...,¿Ver para creer? Acompáñenme a hacer este exp...,2026-05-09,TikTok,Video,kariinacamposs,Costa Rica,7637943869375515924,Huggies,Influencers,...,1.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.077125,0.048128,No Boost,2026-08-07 8:41:09
312,https://www.tiktok.com/@kariinacamposs/video/7...,Cosas que si me han servido en mi maternidad. ...,2026-07-31,TikTok,Video,kariinacamposs,Costa Rica,7668514054863981844,Huggies,Influencers,...,0.625000,0.375000,0.000000,10.000000,6.000000,0.000000,0.077125,0.048128,No Boost,2026-08-07 8:41:17
313,https://www.tiktok.com/@sisoymachis/video/7670...,estamos en mercurio retrógrado o porque todos ...,2026-08-05,TikTok,Video,sisoymachis,Colombia,7670347780652682527,Kotex,Influencers,...,0.522727,0.159091,0.318182,31.363636,9.545455,19.090909,0.135782,0.070111,Review,2026-08-07 8:41:24


In [ ]:
# Save final table
# Open the destination sheets file
sh = gc.open_by_key('1XumLQUH6ApJPiGuT4OMQQ4o6QG_xW1Nev9qZ2aaKIJ0')
worksheet = sh.get_worksheet(0)

# Replace old data with new data
set_with_dataframe(worksheet, tiktok_influencers_final)
print("DataFrame saved successfully!")

DataFrame saved successfully!
